In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-lgd-eval-valid-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3==1.24.59
shap==0.40.0
matplotlib==3.6.1
catboost==1.0.4
seaborn==0.11.2

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import os
import sklearn.metrics as skm
import pandas as pd
import numpy as np
import boto3
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import json

# change wd to absolute path
os.chdir('/tmp')

# eval metrics
def generate_eval_metrics(y_true, y_hat, str_filename, str_dirname_output):
    # dictionary of regression eval metrics
    dict_eval_metrics = {
        'explained_variance': skm.explained_variance_score(y_true=y_true, y_pred=y_hat),
        'mae': skm.mean_absolute_error(y_true=y_true, y_pred=y_hat),
        'mse': skm.mean_squared_error(y_true=y_true, y_pred=y_hat),
        'rmse': np.sqrt(skm.mean_squared_error(y_true=y_true, y_pred=y_hat)),
    }
    # write to .json
    json.dump(dict_eval_metrics, open(f'{str_dirname_output}/{str_filename}', 'w'))
    # return object
    return dict_eval_metrics

# plot distributions of predictions
def plot_distribution_predictions(y_hat_train, y_hat_valid, str_dirname_output, str_filename):
    fig, ax = plt.subplots(figsize=(11,6))
    ax.set_title('Distribution of Predictions in Training and Validation Data')
    ax.set_xlabel('Predicted Loss')
    # train
    sns.kdeplot(y_hat_train, ax=ax, label='Train')
    # valid
    sns.kdeplot(y_hat_valid, ax=ax, label='Valid')
    plt.legend()
    plt.savefig(f'{str_dirname_output}/{str_filename}', bbox_inches='tight')
    plt.close()

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)
    
# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_target = 'target'
    str_dirname_output = '.'

#     # make output dir
#     try:
#         os.mkdir(str_dirname_output)
#     except FileExistsError:
#         pass

    ###############################################################################
    # HYPERPARAMETERS
    ###############################################################################
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))

    # get filename for training
    str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
    print(f'Training filename: {str_filename_train}')

    # get filename for valid
    str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
    print(f'Valid filename: {str_filename_valid}')

    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')

    ##################################################################################

    # import model
    print('Importing best model...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
    df = pd.read_csv(str_uri)
    # get iteration
    int_best_iteration = df['iteration'].iloc[0]

    # get model
    str_filename = f'dict_model_inference_{int_best_iteration}.pkl'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    str_bucket_path = f'03_pricing_lgd/02_model/02_model/02_batch_tuning/models/{str_filename}'
    download_from_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
    dict_pipeline = pickle.load(open(str_local_path, 'rb'))
    cls_model_inference = dict_pipeline['model_inference']
    list_cols_model = list(cls_model_inference.feature_names_)
    list_cols_import = list_cols_model + [str_target]

    # get y_hat_train
    print('Getting y_hat_train...')
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)
    y_hat_train = cls_model_inference.predict(df[list_cols_model])
    y_true_train = df['target']

    # get y_hat_valid
    print('Getting y_hat_valid...')
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)
    y_hat_valid = cls_model_inference.predict(df[list_cols_model])
    y_true_valid = df['target']

    # save memory
    del df

    ##########
    ##########

    # read training data
    print('Reading training data...')
    list_cols = [
        str_target,
    ]
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df = pd.read_parquet(str_uri, columns=list_cols)
    df['y_hat'] = y_hat_train

    # get training eval metrics
    print('Getting training eval metrics...')
    str_filename = 'dict_eval_metrics_train.json'
    dict_eval_metrics = generate_eval_metrics(
        y_true=y_true_train, 
        y_hat=y_hat_train, 
        str_filename=str_filename, 
        str_dirname_output=str_dirname_output,
    )
    str_local_path = f'{str_dirname_output}/{str_filename}'
    str_bucket_path = f'03_pricing_lgd/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

    # lift chart
    df_eval = pd.DataFrame({
        'y_hat': y_hat_train,
        'target': y_true_train,
    })
    # quantile
    df_eval['quantile_rank'] = pd.qcut(df_eval['y_hat'], 20, labels=False, duplicates='drop')
    # group
    df_eval = df_eval.groupby('quantile_rank', as_index=False).agg({
        'target': 'mean',
    })
    # plot
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title('Mean Target (actual) by Prediction Quantile Rank for Train')
    ax.set_xlabel('Quantile Rank')
    ax.set_ylabel('Mean Target')
    ax.plot(df_eval['quantile_rank'].astype(str), df_eval[str_target])
    # save
    str_filename = 'plt_lift_train.png'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    plt.savefig(str_local_path, bbox_inches='tight')
    # upload
    str_bucket_path = f'03_pricing_lgd/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

    # save memory
    del df

    ##########
    ##########

    # read validation data
    print('Reading validation data...')
    list_cols = [
        str_target,
    ]
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df = pd.read_parquet(str_uri, columns=list_cols)
    df['y_hat'] = y_hat_valid

    # get validation eval metrics
    print('Getting validation eval metrics...')
    str_filename = 'dict_eval_metrics_valid.json'
    dict_eval_metrics = generate_eval_metrics(
        y_true=y_true_valid, 
        y_hat=y_hat_valid, 
        str_filename=str_filename, 
        str_dirname_output=str_dirname_output,
    )
    str_local_path = f'{str_dirname_output}/{str_filename}'
    str_bucket_path = f'03_pricing_lgd/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

    # lift chart
    df_eval = pd.DataFrame({
        'y_hat': y_hat_valid,
        'target': y_true_valid,
    })
    # quantile
    df_eval['quantile_rank'] = pd.qcut(df_eval['y_hat'], 20, labels=False, duplicates='drop')
    # group
    df_eval = df_eval.groupby('quantile_rank', as_index=False).agg({
        'target': 'mean',
    })
    # plot
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title('Mean Target (actual) by Prediction Quantile Rank for Valid')
    ax.set_xlabel('Quantile Rank')
    ax.set_ylabel('Mean Target')
    ax.plot(df_eval['quantile_rank'].astype(str), df_eval[str_target])
    # save
    str_filename = 'plt_lift_valid.png'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    plt.savefig(str_local_path, bbox_inches='tight')
    # upload
    str_bucket_path = f'03_pricing_lgd/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

    ##########
    ##########

    # distribution of predictions
    print('Plottiong distribution of predictions...')
    str_filename = 'plt_dist_yhat.png'
    plot_distribution_predictions(
        y_hat_train=y_hat_train, 
        y_hat_valid=y_hat_valid, 
        str_dirname_output=str_dirname_output, 
        str_filename=str_filename,
    )
    str_local_path = f'{str_dirname_output}/{str_filename}'
    str_bucket_path = f'03_pricing_lgd/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-lgd-eval-valid-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon   85.5kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> a00d7145cc1d
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 2b238a4283cb
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> e7d10febc96f
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> d3c8ee1b3cce
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> Using cache
 ---> 4a2df16b8264
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Using cache
 ---> f932f4a842b5
Successfully built f932f4a842b5
Successfully tagged genxii-lgd-eval-valid-2:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-lgd-eval-valid-2' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-eval-valid-2]
bfc835d6fcda: Preparing
9498babfbd8b: Preparing
f7ebc8651c29: Preparing
dbd3c2db499f: Preparing
45d114d35c45: Preparing
86e4d644d316: Preparing
15dd6c63f3a2: Preparing
dd00ec5a5244: Preparing
8f43d000b361: Preparing
c349004e3af3: Preparing
86e4d644d316: Waiting
c349004e3af3: Waiting
8f43d000b361: Waiting
dd00ec5a5244: Waiting
15dd6c63f3a2: Waiting
f7ebc8651c29: Layer already exists
9498babfbd8b: Layer already exists
bfc835d6fcda: Layer already exists
dbd3c2db499f: Layer already exists
45d114d35c45: Layer already exists
86e4d644d316: Layer already exists
dd00ec5a5244: Layer already exists
15dd6c63f3a2: Layer already exists
c349004e3af3: Layer already exists
8f43d000b361: Layer already exists
latest: digest: sha256:10629d6245ad4089f81c700c109943acd45d674e5346e90f9aaa3b5455f415ec size: 2420


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 08 Nov 2023 23:13:10 GMT',
                                      'x-amzn-requestid': 'd344db1a-d73a-48fd-88b1-8c5ea4819b38'},
                      'HTTPStatusCode': 204,
                      'RequestId': 'd344db1a-d73a-48fd-88b1-8c5ea4819b38',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900,
    MemorySize=1000,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '10629d6245ad4089f81c700c109943acd45d674e5346e90f9aaa3b5455f415ec',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-eval-valid-2',
 'FunctionName': 'genxii-lgd-eval-valid-2',
 'LastModified': '2023-11-08T23:13:10.353+0000',
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1069',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 08 Nov 2023 23:13:11 GMT',
                                      'x-amzn-requestid': 'a0bcb168-0b3f-4199-b373-5b2f3e90f5e0'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'a0bcb168-0b3f-4199-b373-5b2f3e90f5e0',
                      'RetryAttempts': 0},
 'RevisionId': 'b480827d-959d-4061-8155-

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)